In [7]:
from pathlib import Path

import torch
import torch.nn as nn 

from src.data.load_data import load_movielens

In [10]:
DATA_DIR = Path('data/raw/ml-1m')

ratings, movies = load_movielens(DATA_DIR)

In [14]:
user_id = 1

user_history = (
    ratings[ratings["user_id"] == user_id]
    .sort_values("timestamp")["movie_id"]
    .tolist()
)

user_history[:10]

[3186, 1721, 1022, 1270, 2340, 1836, 3408, 1207, 2804, 720]

In [15]:
movie_ids = torch.tensor(
    user_history[:5],
    dtype=torch.long
) 

movie_ids

tensor([3186, 1721, 1022, 1270, 2340])

In [17]:
movie_ids = sorted(ratings['movie_id'].unique())

movie2idx = {
    movie_id: idx
    for idx, movie_id in enumerate(movie_ids)
}

idx2movie = {
    idx: movie_id
    for movie_id, idx in enumerate(movie_ids)
}

len(movie2idx)

3706

In [26]:
ratings["movie_idx"] = ratings["movie_id"].map(movie2idx)

In [22]:
ratings.head()

,user_id,movie_id,rating,timestamp,movie_idx
0,1,1193,5,978300760,1104
1,1,661,3,978302109,639
2,1,914,3,978301968,853
3,1,3408,4,978300275,3177
4,1,2355,5,978824291,2162


In [23]:
movie_ids = torch.tensor(
    ratings.loc[
        ratings["user_id"] == 1,
        "movie_idx"
    ].head(5).to_numpy(),
    dtype=torch.long
)

In [24]:
embedding = nn.Embedding(
    num_embeddings=len(movie2idx),
    embedding_dim=128
)

In [36]:
from src.models.embedding import MovieEmbedding

embedding = MovieEmbedding(
    num_movies=len(movie2idx),
    embedding_dim=128
)


In [38]:
movie_vectors = embedding(movie_ids)
movie_vectors.shape

torch.Size([5, 128])

In [40]:
from src.models.positional_embedding import PositionalEmbedding

position_embedding = PositionalEmbedding(
    max_seq_len=50,
    embedding_dim=128,
)

positions = torch.arange(5)

positions

tensor([0, 1, 2, 3, 4])